
* Before run download retail_store_sales.csv

# 1. Загрузка и предварительная обработка данных

## 1.1. Загрузка и вывод схемы

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Инициализация SparkSession
spark = (SparkSession.builder 
    .appName("FinalSparkTask") 
    .getOrCreate())

try:
    df = spark.read.csv('retail_store_sales.csv', header=True) # inferSchema=True не указывал для 1.3
    print("Data:")
    print(df.show(5))
    # or
    # print(df.take(5))
    print("Schema:")
    print(df.printSchema())
except Exception as ex:
    raise ex

Data:
+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|Transaction ID|Customer ID|     Category|        Item|Price Per Unit|Quantity|Total Spent|Payment Method|Location|Transaction Date|Discount Applied|
+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|   Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|   TXN_3731986|    CUST_22|Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|   TXN_9303719|    CUST_02|     Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|   TXN_9458126|    CUST_06|    Beverages| Item_16_BEV|          27.5|     9.0|      247.5|   

## 1.2. Очистка названий столбцов: Преобразуйте названия всех столбцов к единому регистру - snake_case.  Выведите обновленную схему DataFrame  или названия столбцов, чтобы убедиться в изменении названий.

In [2]:
# Добавим функцию перевода слова в snake_case
import re
def to_snake_case(column_name: str) -> str:
    column_name.replace('-', ' ')
    column_name = re.sub('([A-Z]+)', r' \1', column_name)
    column_name = re.sub('([a-z])([A-Z])', r'\1 \2', column_name)
    return '_'.join(column_name.split()).lower()

In [3]:
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, to_snake_case(col_name))
df.show(5)

+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|transaction_id|customer_id|     category|        item|price_per_unit|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|   Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|   TXN_3731986|    CUST_22|Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|   TXN_9303719|    CUST_02|     Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|   TXN_9458126|    CUST_06|    Beverages| Item_16_BEV|          27.5|     9.0|      247.5|   Credit

## 1.3. Преобразование типов данных: Проанализируйте к каким типам данных относятся данные в столбцах и приведите столбец к соответствующему типу. 

* Надо перевести во float price_per_unit, quantity, total_spent
* Надо перевести в date transaction_date
* Надо перевести в bool discount_applied

In [4]:
df = df.withColumn(
    "price_per_unit", F.col("price_per_unit").cast("float")
     ).withColumn(
    "quantity", F.col("quantity").cast("float")
     ).withColumn(
    "total_spent", F.col("total_spent").cast("float")
     ).withColumn(
    "transaction_date", F.to_date(F.col("transaction_date"), "yyyy-MM-dd")
     ).withColumn(
    "discount_applied", F.col("discount_applied").cast("boolean")
     )


In [5]:
print("Data:")
print(df.show(5))
print("Schema:")
print(df.printSchema())

Data:
+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|transaction_id|customer_id|     category|        item|price_per_unit|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------+-----------+-------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|   Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            true|
|   TXN_3731986|    CUST_22|Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            true|
|   TXN_9303719|    CUST_02|     Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           false|
|   TXN_9458126|    CUST_06|    Beverages| Item_16_BEV|          27.5|     9.0|      247.5|   

# 2. Очистка и валидация данных

## 2.1 Заполнение отсутствующие Price Per Unit

In [6]:
df_2_1_step = df.withColumn(
    "price_per_unit",
    F.when(
        F.col("price_per_unit").isNull() & ~F.col("quantity").isNull() & ~F.col("total_spent").isNull(),
        F.col("total_spent") / F.col("quantity")
    )
    .otherwise(F.col("price_per_unit"))
)

In [7]:
df.select("price_per_unit", "quantity", "total_spent").show()

+--------------+--------+-----------+
|price_per_unit|quantity|total_spent|
+--------------+--------+-----------+
|          18.5|    10.0|      185.0|
|          29.0|     9.0|      261.0|
|          21.5|     2.0|       43.0|
|          27.5|     9.0|      247.5|
|          12.5|     7.0|       87.5|
|          NULL|    10.0|      200.0|
|           5.0|     8.0|       40.0|
|          33.5|    NULL|       NULL|
|          27.5|     1.0|       27.5|
|          36.5|     3.0|      109.5|
|           8.0|     9.0|       72.0|
|          NULL|     8.0|       52.0|
|           6.5|     7.0|       45.5|
|          39.5|     6.0|      237.0|
|          27.5|     2.0|       55.0|
|          24.5|    NULL|       NULL|
|          29.0|     8.0|      232.0|
|          NULL|    10.0|      275.0|
|          23.0|     1.0|       23.0|
|          35.0|    NULL|       NULL|
+--------------+--------+-----------+
only showing top 20 rows



In [8]:
df_2_1_step.select("price_per_unit", "quantity", "total_spent").show()

+--------------+--------+-----------+
|price_per_unit|quantity|total_spent|
+--------------+--------+-----------+
|          18.5|    10.0|      185.0|
|          29.0|     9.0|      261.0|
|          21.5|     2.0|       43.0|
|          27.5|     9.0|      247.5|
|          12.5|     7.0|       87.5|
|          20.0|    10.0|      200.0|
|           5.0|     8.0|       40.0|
|          33.5|    NULL|       NULL|
|          27.5|     1.0|       27.5|
|          36.5|     3.0|      109.5|
|           8.0|     9.0|       72.0|
|           6.5|     8.0|       52.0|
|           6.5|     7.0|       45.5|
|          39.5|     6.0|      237.0|
|          27.5|     2.0|       55.0|
|          24.5|    NULL|       NULL|
|          29.0|     8.0|      232.0|
|          27.5|    10.0|      275.0|
|          23.0|     1.0|       23.0|
|          35.0|    NULL|       NULL|
+--------------+--------+-----------+
only showing top 20 rows



## 2.2. Восстановление отсутствующих Item

In [9]:
product_directory_df = df_2_1_step.select("item", "category", "price_per_unit").filter(df_2_1_step.item.isNotNull()).distinct()
print(f"Количество уникальных названий товаров: {product_directory_df.count()}")
print("Дополнительная проверка, чтобы не было повторений в названиях:")
if product_directory_df.select("item").distinct().count() == product_directory_df.count():
    print("Успешно")
else:
    print("Не успешно")
product_directory_df.show()

Количество уникальных названий товаров: 200
Дополнительная проверка, чтобы не было повторений в названиях:
Успешно
+------------+--------------------+--------------+
|        item|            category|price_per_unit|
+------------+--------------------+--------------+
|Item_11_FOOD|                Food|          20.0|
| Item_21_PAT|          Patisserie|          35.0|
| Item_10_FUR|           Furniture|          18.5|
| Item_13_FUR|           Furniture|          23.0|
|  Item_2_PAT|          Patisserie|           6.5|
| Item_22_PAT|          Patisserie|          36.5|
|  Item_3_FUR|           Furniture|           8.0|
| Item_16_BUT|            Butchers|          27.5|
| Item_20_BEV|           Beverages|          33.5|
| Item_18_BUT|            Butchers|          30.5|
| Item_7_MILK|       Milk Products|          14.0|
|  Item_5_BEV|           Beverages|          11.0|
| Item_25_PAT|          Patisserie|          41.0|
| Item_12_PAT|          Patisserie|          21.5|
|  Item_6_EHE|Elec

In [10]:
df_2_2_step = df_2_1_step.join(
    product_directory_df, 
     ['category', 'price_per_unit'], 
    "left", # Берем все строки из df_2_1_step
            # Где нет category или price_per_unit, значения product_directory_df.item так же заменятся на NULL
).drop(df_2_1_step.item)

In [11]:
df_2_1_step.select("transaction_id", "item", "price_per_unit", "category").sort(df_2_1_step.transaction_id).show(10)

+--------------+------------+--------------+--------------------+
|transaction_id|        item|price_per_unit|            category|
+--------------+------------+--------------+--------------------+
|   TXN_1002182| Item_5_FOOD|          11.0|                Food|
|   TXN_1003865|  Item_2_FUR|           6.5|           Furniture|
|   TXN_1003940|  Item_5_FUR|          11.0|           Furniture|
|   TXN_1004091|Item_25_FOOD|          41.0|                Food|
|   TXN_1004124|  Item_7_CEA|          14.0|Computers and ele...|
|   TXN_1004284|Item_25_MILK|          41.0|       Milk Products|
|   TXN_1005543|        NULL|          30.5|                Food|
|   TXN_1005750| Item_12_EHE|          21.5|Electric househol...|
|   TXN_1006123|  Item_8_EHE|          15.5|Electric househol...|
|   TXN_1006129|Item_17_MILK|          29.0|       Milk Products|
+--------------+------------+--------------+--------------------+
only showing top 10 rows



In [12]:
df_2_2_step.select("transaction_id", "item", "price_per_unit", "category").sort(df_2_2_step.transaction_id).show(10)

+--------------+------------+--------------+--------------------+
|transaction_id|        item|price_per_unit|            category|
+--------------+------------+--------------+--------------------+
|   TXN_1002182| Item_5_FOOD|          11.0|                Food|
|   TXN_1003865|  Item_2_FUR|           6.5|           Furniture|
|   TXN_1003940|  Item_5_FUR|          11.0|           Furniture|
|   TXN_1004091|Item_25_FOOD|          41.0|                Food|
|   TXN_1004124|  Item_7_CEA|          14.0|Computers and ele...|
|   TXN_1004284|Item_25_MILK|          41.0|       Milk Products|
|   TXN_1005543|Item_18_FOOD|          30.5|                Food|
|   TXN_1005750| Item_12_EHE|          21.5|Electric househol...|
|   TXN_1006123|  Item_8_EHE|          15.5|Electric househol...|
|   TXN_1006129|Item_17_MILK|          29.0|       Milk Products|
+--------------+------------+--------------+--------------------+
only showing top 10 rows



## 2.3. Заполнение отсутствующих Quantity и Total Spent

* По проверкам ниже видно, что строк для изменения не осталось 

In [13]:
df_2_2_step.filter(
    df_2_2_step.total_spent.isNull() & df_2_2_step.quantity.isNotNull()
).show()

+--------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+----+
|category|price_per_unit|transaction_id|customer_id|quantity|total_spent|payment_method|location|transaction_date|discount_applied|item|
+--------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+----+
+--------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+----+



In [14]:
df_2_2_step.filter(
    df_2_2_step.quantity.isNull() & df_2_2_step.total_spent.isNotNull()
).show()

+--------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+----+
|category|price_per_unit|transaction_id|customer_id|quantity|total_spent|payment_method|location|transaction_date|discount_applied|item|
+--------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+----+
+--------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+----+



* Алгоритм для них был бы такой, если бы они были

In [15]:
df_2_3_step = df_2_2_step.withColumn(
    "total_spent",
    F.when(
        F.col("total_spent").isNull() & ~F.col("quantity").isNull() & ~F.col("price_per_unit").isNull(),
        F.col("price_per_unit") * F.col("quantity")
    )
    .otherwise(F.col("total_spent"))
).withColumn(
    "quantity",
    F.when(
        F.col("quantity").isNull() & ~F.col("price_per_unit").isNull() & ~F.col("total_spent").isNull(),
        F.round(F.col("total_spent") / F.col("price_per_unit"))
    )
    .otherwise(F.col("quantity"))
)

In [16]:
df_2_3_step.show()

+--------------------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+------------+
|            category|price_per_unit|transaction_id|customer_id|quantity|total_spent|payment_method|location|transaction_date|discount_applied|        item|
+--------------------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+------------+
|          Patisserie|          18.5|   TXN_6867343|    CUST_09|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            true| Item_10_PAT|
|       Milk Products|          29.0|   TXN_3731986|    CUST_22|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            true|Item_17_MILK|
|            Butchers|          21.5|   TXN_9303719|    CUST_02|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           false| Item_12_BUT|
|           Beverages|          27.5|   TXN_9458126|    CU

## 2.4. Удалите оставшийся строки с пропусками в Category, Quantity ,Total Spent и Price Per Unit

In [17]:
df_2_4_step = df_2_3_step.dropna(subset=['price_per_unit', 'quantity', 'total_spent', 'category'])
df_2_4_step.show()

+--------------------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+------------+
|            category|price_per_unit|transaction_id|customer_id|quantity|total_spent|payment_method|location|transaction_date|discount_applied|        item|
+--------------------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+------------+
|          Patisserie|          18.5|   TXN_6867343|    CUST_09|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            true| Item_10_PAT|
|       Milk Products|          29.0|   TXN_3731986|    CUST_22|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            true|Item_17_MILK|
|            Butchers|          21.5|   TXN_9303719|    CUST_02|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           false| Item_12_BUT|
|           Beverages|          27.5|   TXN_9458126|    CU

In [18]:
print(f"Было: {df_2_3_step.count()} строк")
print(f"Стало: {df_2_4_step.count()} строк")

Было: 12575 строк
Стало: 11971 строк


# 3. Разведочный анализ данных

## 3.1. Самые популярные категории товаров

In [24]:
# 3.1. Самые популярные категории товаров: Рассчитайте общее количество проданных единиц товара  для каждой категории. Определите Топ-5 категорий по общему количеству проданных единиц. 

df_3_1_step = df_2_4_step.groupBy("category").agg(
    F.sum("quantity").alias("quantity_total_products"),
)
df_3_1_step.show()

+--------------------+-----------------------+
|            category|quantity_total_products|
+--------------------+-----------------------+
|                Food|                 8387.0|
|Computers and ele...|                 8272.0|
|          Patisserie|                 7943.0|
|           Beverages|                 8358.0|
|Electric househol...|                 8309.0|
|       Milk Products|                 8339.0|
|           Furniture|                 8462.0|
|            Butchers|                 8206.0|
+--------------------+-----------------------+



* Топ 5 по количеству проданных товаров

In [25]:
df_3_1_step.orderBy(df_3_1_step.quantity_total_products.desc()).show(5, truncate=False)

+-----------------------------+-----------------------+
|category                     |quantity_total_products|
+-----------------------------+-----------------------+
|Furniture                    |8462.0                 |
|Food                         |8387.0                 |
|Beverages                    |8358.0                 |
|Milk Products                |8339.0                 |
|Electric household essentials|8309.0                 |
+-----------------------------+-----------------------+
only showing top 5 rows



## 3.2. Анализ среднего чека

* Рассчитайте среднее значение Total Spent для каждого метода оплаты. Округлите до двух знаков после запятой.
* Рассчитайте среднее значение Total Spent для каждой места где прошла оплата. Округлите до двух знаков после запятой.



In [29]:
df_3_2_step = df_2_4_step.groupBy("payment_method").agg(
    F.round(F.avg("total_spent"), 2).alias("mean_total_spent"),
)
df_3_2_step.show()

+--------------+----------------+
|payment_method|mean_total_spent|
+--------------+----------------+
|   Credit Card|          129.13|
|Digital Wallet|          128.72|
|          Cash|          131.05|
+--------------+----------------+



In [30]:
df_3_2_step_2 = df_2_4_step.groupBy("location").agg(
    F.round(F.avg("total_spent"), 2).alias("mean_total_spent"),
)
df_3_2_step_2.show()

+--------+----------------+
|location|mean_total_spent|
+--------+----------------+
|In-store|          128.86|
|  Online|          130.42|
+--------+----------------+



# 4. Генерация признаков 

## 4.1. Временные признаки

In [32]:
df_4_1_step = df_2_4_step.withColumn(
    # Извлекаем год из даты начала курса
    "day_of_week", F.dayofweek(F.col("transaction_date"))
).withColumn(
    # Извлекаем месяц из даты начала курса
    "transaction_month", F.month(F.col("transaction_date"))
)
df_4_1_step.show()

# 2024-04-08 день недели - понедельник, но в америке вроде с воскресенья отсчет начинается

+--------------------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+------------+-----------+-----------------+
|            category|price_per_unit|transaction_id|customer_id|quantity|total_spent|payment_method|location|transaction_date|discount_applied|        item|day_of_week|transaction_month|
+--------------------+--------------+--------------+-----------+--------+-----------+--------------+--------+----------------+----------------+------------+-----------+-----------------+
|          Patisserie|          18.5|   TXN_6867343|    CUST_09|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            true| Item_10_PAT|          2|                4|
|       Milk Products|          29.0|   TXN_3731986|    CUST_22|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            true|Item_17_MILK|          1|                7|
|            Butchers|          21.5|   TXN_9303719|    CUST_02| 

## 4.2. Продажи по дням недели

In [35]:
df_4_2_step = df_4_1_step.groupBy("day_of_week").agg(
    F.round(F.avg("total_spent"), 2).alias("mean_total_spent"),
)
df_4_2_step.orderBy(df_4_2_step.day_of_week).show(7)

+-----------+----------------+
|day_of_week|mean_total_spent|
+-----------+----------------+
|          1|          130.18|
|          2|          125.57|
|          3|          129.51|
|          4|          126.82|
|          5|          129.28|
|          6|          134.64|
|          7|          131.49|
+-----------+----------------+



## 4.3.Продажи по месяцам

In [37]:
df_4_3_step = df_4_1_step.groupBy("transaction_month").agg(
    F.round(F.avg("total_spent"), 2).alias("mean_total_spent"),
)
df_4_3_step.orderBy(df_4_3_step.transaction_month).show(12)

+-----------------+----------------+
|transaction_month|mean_total_spent|
+-----------------+----------------+
|                1|          134.69|
|                2|          130.66|
|                3|          126.83|
|                4|          131.81|
|                5|           127.4|
|                6|          130.95|
|                7|          126.57|
|                8|          124.28|
|                9|          131.45|
|               10|          127.85|
|               11|          128.79|
|               12|          133.15|
+-----------------+----------------+



## 4.4. Признаки клиента

In [38]:
df_4_4_step = df_4_1_step.groupBy("customer_id").agg(
    F.round(F.sum("total_spent"), 2).alias("customer_lifetime_value"),
)
df_4_4_step.orderBy(df_4_4_step.customer_lifetime_value.desc()).show(10)

+-----------+-----------------------+
|customer_id|customer_lifetime_value|
+-----------+-----------------------+
|    CUST_24|                68452.0|
|    CUST_08|                67351.5|
|    CUST_05|                66974.5|
|    CUST_16|                65570.5|
|    CUST_13|                65037.0|
|    CUST_23|                64507.0|
|    CUST_10|                63155.5|
|    CUST_15|                63117.5|
|    CUST_21|                62933.0|
|    CUST_02|                62046.5|
+-----------+-----------------------+
only showing top 10 rows

